# Test 3b — Multi-Seed Robustness, **DFG arm** — seed 123One seed per Kaggle session. A cold-start DFG run is heavier than the text arm(512 positions plus a 512x512 attention mask per sample), so budget a full session.**Order: 42, 123, 2025, 7, 2718.** Run 42 first; it prints a pre-flight block thatfails closed on the partition and reports DFG statistics before any training happens.## Why this existsTest 3 varied the seed on the **text** arm only and compared five runs against asingle DFG checkpoint. That bounds the text arm's distribution around a fixedpoint; it does not compare two distributions. These five runs supply the missingarm so the comparison can be made **paired**:    delta_s = accuracy(text, seed s) - accuracy(DFG, seed s)## Pre-registration — fixed before any run* **n = 5**, seeds 42, 123, 2025, 7, 2718 — the same seeds as the text arm, so the  differences pair.* **Primary analysis**: paired t-test on the five per-seed differences, plus TOST  equivalence at the **±0.25 pp** bound already reported in Section V-D.* **Reported at n = 5 regardless of outcome.** Adding seeds after seeing the result  would be optional stopping and would void the test.* A result that *reverses* the current finding is reported as such.## ConfigurationMatched to `training_notebooks/re_train/graphcodebert-train-dfg.ipynb`, the notebookthat produced Table 2's 87.8593%, except where noted.| | value ||---|---|| start | `microsoft/graphcodebert-base` (cold) || code / DFG budget | 384 tokens + 128 nodes || micro-batch x accumulation | 16 x 2 = **effective 32** || epochs / patience | 10 / 2 || warmup | 10% of total steps || partition | split seed 42 fixed, duplicate-filtered to 18,541 |**One deliberate difference from that notebook.** It sets`cudnn.deterministic = True`, which the text-arm seed runs did not. Leaving it onwould give this arm less run-to-run noise than the arm it is compared against, andthe whole point is to compare their spreads. Both arms therefore run without it.## Kaggle setup    + Add Input -> Datasets -> the dfgdataset2 corpus    GPU on, internet on (downloads microsoft/graphcodebert-base)

In [ ]:
# ============================================================#  SEED 123  --  SECOND of five. One seed per Kaggle session.#  DFG ARM. Pairs with test3_seed123 from the text arm.# ============================================================SEED = 123# ============================================================!pip install transformers -q

In [ ]:
import os, json, math, random, hashlib, timeimport numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import DataLoader, Dataset, Subset, SequentialSampler, RandomSamplerfrom torch.optim import AdamWfrom torch.amp import autocast, GradScalerfrom transformers import AutoTokenizer, RobertaConfig, RobertaModel, get_linear_schedule_with_warmupfrom collections import defaultdictfrom tqdm.auto import tqdmclass Args:    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"    model_name_or_path = "microsoft/graphcodebert-base"   # COLD START. Not our checkpoint.    code_length      = 384      # matches graphcodebert-train-dfg.ipynb    data_flow_length = 128      # DFG node budget    train_batch_size            = 16    gradient_accumulation_steps = 2     # effective batch 32, as in the DFG arm    eval_batch_size             = 32    learning_rate     = 2e-5    max_grad_norm     = 1.0    num_train_epochs  = 10    patience          = 2    time_budget_hours = 11.0    # predictive check below; Kaggle wall is 12h    test_ratio = 0.10    val_ratio  = 0.08    split_seed = 42             # the PARTITION is fixed; only SEED varies    num_workers = 2    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")args = Args()args.seed = SEEDargs.total_len = args.code_length + args.data_flow_lengthprint(f"Seed        : {args.seed}")print(f"Device      : {args.device}  ({torch.cuda.device_count()} GPU)")print(f"Start from  : {args.model_name_or_path}  <- public weights, cold start")print(f"Budget      : {args.code_length} code + {args.data_flow_length} DFG = {args.total_len} positions")print(f"Batch       : {args.train_batch_size} x {args.gradient_accumulation_steps}"      f" = {args.train_batch_size * args.gradient_accumulation_steps} effective")print(f"Ceiling     : {args.num_train_epochs} epochs, patience {args.patience}")

In [ ]:
# Verbatim from test_scripts/split_and_filter.py, and identical to the text-arm# seed notebooks. infer_source has no filename fallback: the corpus carries no# source key, so every record resolves to "unknown" and the split is one shuffle.# Adding a fallback rebuilds Partition S and its 89.9% leak (PAPER.md 5.2).def infer_source(entry):    for key in ("source", "dataset", "origin", "project"):        v = entry.get(key)        if v is not None and str(v).strip() != "":            return str(v).strip()    return "unknown"def allocate_counts(total_needed, groups, fraction):    raw  = {g: len(v) * fraction for g, v in groups.items()}    base = {g: int(math.floor(v)) for g, v in raw.items()}    rem  = total_needed - sum(base.values())    order = sorted(groups, key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)    for g in order[:rem]:        base[g] += 1    return basedef get_split_indices(filepath, test_ratio, val_ratio, seed):    srcs, hashes = [], []    with open(filepath, 'r', encoding='utf-8') as f:        for line in f:            e = json.loads(line)            srcs.append(infer_source(e))            hashes.append(hashlib.md5(str(e.get('code','')).encode('utf-8','ignore')).hexdigest())            del e    total = len(srcs)    rng = random.Random(seed)    groups = defaultdict(list)    for i, s in enumerate(srcs):        groups[s].append(i)    for v in groups.values():        rng.shuffle(v)    t_alloc = allocate_counts(int(round(total * test_ratio)), groups, test_ratio)    rest, test_idx = {}, []    for s, idx in groups.items():        k = min(t_alloc[s], len(idx))        test_idx.extend(idx[:k]); rest[s] = idx[k:]    v_alloc = allocate_counts(int(round(total * val_ratio)), rest, val_ratio / (1.0 - test_ratio))    val_idx, train_idx = [], []    for s, idx in rest.items():        k = min(v_alloc[s], len(idx))        val_idx.extend(idx[:k]); train_idx.extend(idx[k:])    train_idx, val_idx, test_idx = sorted(train_idx), sorted(val_idx), sorted(test_idx)    assert set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)    print(f"Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")    seen = {hashes[i] for i in train_idx}    seen.update(hashes[i] for i in val_idx)    before = len(test_idx)    test_idx = [i for i in test_idx if hashes[i] not in seen]    print(f"Duplicate filter: dropped {before-len(test_idx):,} "          f"({(before-len(test_idx))/before:.2%}) -> {len(test_idx):,} clean")    return train_idx, val_idx, test_idxtrain_idx, val_idx, test_idx = get_split_indices(    args.train_file, args.test_ratio, args.val_ratio, args.split_seed)# Fail closed. Every number in this project uses these three sizes, and the text# arm this run pairs with used exactly them.assert (len(train_idx), len(val_idx), len(test_idx)) == (163967, 15997, 18541), (    f"partition mismatch: {len(train_idx)}/{len(val_idx)}/{len(test_idx)} "    f"!= 163967/15997/18541 -- results would not pair with the text arm")print("Partition matches the text-arm seed runs exactly.")

In [ ]:
def set_seed(s):    random.seed(s); np.random.seed(s); torch.manual_seed(s)    if torch.cuda.device_count() > 0:        torch.cuda.manual_seed_all(s)    # Deliberately NOT setting cudnn.deterministic. See the header: the text arm    # ran without it, and forcing it here would give this arm less run-to-run    # noise than the arm it is being compared against.set_seed(args.seed)class Model(nn.Module):    """Graph-guided masked attention, verbatim from the DFG training notebook."""    def __init__(self, encoder, config):        super().__init__()        self.encoder = encoder        self.config = config        self.dropout = nn.Dropout(config.hidden_dropout_prob)        self.classifier = nn.Linear(config.hidden_size, 2)    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):        extended_attention_mask = ((1.0 - attn_mask) * -10000.0).unsqueeze(1)        embedding_output = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)        encoder_outputs = self.encoder.encoder(            embedding_output,            attention_mask=extended_attention_mask,            head_mask=[None] * self.config.num_hidden_layers)        sequence_output = encoder_outputs[0]        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))        prob = F.softmax(logits, dim=-1)        if labels is not None:            return nn.CrossEntropyLoss()(logits, labels), prob        return probclass DFGDataset(Dataset):    """Verbatim from the DFG training notebook, restricted to a set of indices."""    def __init__(self, tokenizer, args, file_path, indices):        self.args = args        self.tokenizer = tokenizer        self.total_len = args.total_len        with open(file_path, 'r', encoding='utf-8') as f:            all_lines = f.readlines()        self.lines = [all_lines[i] for i in indices]        del all_lines    def __len__(self):        return len(self.lines)    def _get_char_index(self, code_lines, coord):        row, col = coord        char_idx = 0        for i in range(min(row, len(code_lines))):            char_idx += len(code_lines[i])        return char_idx + col    def __getitem__(self, item):        entry = json.loads(self.lines[item])        code  = entry.get('code', '')        dfg   = (entry.get('dfg') or [])[:self.args.data_flow_length]        label = int(entry.get('label', 0) or 0)        tok = self.tokenizer(code, max_length=self.args.code_length, truncation=True,                             padding='max_length', return_offsets_mapping=True)        input_ids = tok['input_ids']        offsets   = tok['offset_mapping']        code_lines = code.splitlines(keepends=True)        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)        pos_to_node_idx, node_to_token_map = {}, {}        for node_idx, it in enumerate(dfg):            start_pos, end_pos = it[1][0], it[1][1]            pos_to_node_idx[(start_pos[0], start_pos[1], end_pos[0], end_pos[1])] = node_idx            char_start = self._get_char_index(code_lines, start_pos)            char_end   = self._get_char_index(code_lines, end_pos)            aligned = []            for t_idx, (t_s, t_e) in enumerate(offsets):                if t_s == t_e:                    continue                if (t_s >= char_start and t_e <= char_end) or (char_start >= t_s and char_end <= t_e):                    aligned.append(t_idx)            node_to_token_map[node_idx] = aligned        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)        c_len = self.args.code_length        attn_mask[:c_len, :c_len] = True        for node_idx, it in enumerate(dfg):            abs_node = c_len + node_idx            for t_idx in node_to_token_map.get(node_idx, []):                attn_mask[abs_node, t_idx] = True                attn_mask[t_idx, abs_node] = True            for p_pos in it[4]:                p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])                if p_key in pos_to_node_idx:                    abs_parent = c_len + pos_to_node_idx[p_key]                    attn_mask[abs_node, abs_parent] = True                    attn_mask[abs_parent, abs_node] = True            attn_mask[abs_node, abs_node] = True        full_input_ids = input_ids + dfg_ids        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)        pad = self.total_len - len(full_input_ids)        if pad > 0:            full_input_ids += [self.tokenizer.pad_token_id] * pad            p_ids += [1] * pad        return {'input_ids': torch.tensor(full_input_ids, dtype=torch.long),                'p_ids':     torch.tensor(p_ids, dtype=torch.long),                'attn_mask': torch.tensor(attn_mask, dtype=torch.float),                'label':     torch.tensor(label, dtype=torch.long)}print("Loading tokenizer + datasets ...")tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)train_ds = DFGDataset(tokenizer, args, args.train_file, train_idx)val_ds   = DFGDataset(tokenizer, args, args.train_file, val_idx)test_ds  = DFGDataset(tokenizer, args, args.train_file, test_idx)print(f"  train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_ds):,}")

In [ ]:
# ── PRE-FLIGHT ──────────────────────────────────────────────────────────────# The silent failure this guards against: if the DFG never reaches the model,# this run is a text-only run wearing a DFG label, and it would look plausible.# Nothing below trains; it costs seconds and is worth it at 10 GPU-hours a run.print("Pre-flight checks")print("-" * 62)_n_probe = min(300, len(train_ds.lines))_nodes, _empty = [], 0for _i in range(_n_probe):    _e = json.loads(train_ds.lines[_i])    _d = _e.get('dfg') or []    _nodes.append(len(_d))    if len(_d) == 0:        _empty += 1_nodes.sort()print(f"  DFG nodes over {_n_probe} training records:")print(f"    empty        : {_empty} ({_empty/_n_probe:.1%})")print(f"    median / mean: {_nodes[_n_probe//2]} / {sum(_nodes)/_n_probe:.1f}")print(f"    at the {args.data_flow_length}-node cap: "      f"{sum(1 for c in _nodes if c >= args.data_flow_length)/_n_probe:.1%}")assert _empty / _n_probe < 0.10, "over 10% of records carry no DFG -- check the corpus"_b = train_ds[0]_m = _b['attn_mask']_c = args.code_length_dense = _m.mean().item()_node_block = _m[_c:, _c:]print(f"\n  attention mask on one sample:")print(f"    shape            : {tuple(_m.shape)}  (expect {args.total_len} x {args.total_len})")print(f"    fraction open    : {_dense:.3f}")print(f"    node-node block  : {_node_block.mean().item():.4f} open")print(f"    code-node links  : {int(_m[:_c, _c:].sum().item())} entries")assert tuple(_m.shape) == (args.total_len, args.total_len), "mask is the wrong shape"assert _dense < 0.95, "mask is almost fully open -- the graph is not restricting attention"assert _m[:_c, _c:].sum().item() > 0, "no code-to-node links -- DFG is not connected to the code"print(f"    p_ids: code {_b['p_ids'][:3].tolist()} ... nodes "      f"{_b['p_ids'][_c:_c+3].tolist()} (nodes must be 0)")assert _b['p_ids'][_c].item() == 0, "DFG nodes are not at position 0"print("\n  All pre-flight checks passed. The DFG reaches the model.")print("-" * 62)

In [ ]:
@torch.no_grad()def evaluate(model, ds, desc):    model.eval()    loader = DataLoader(ds, sampler=SequentialSampler(ds),                        batch_size=args.eval_batch_size, num_workers=args.num_workers)    probs, labels = [], []    for batch in tqdm(loader, desc=desc, leave=False):        with autocast(device_type="cuda", enabled=torch.cuda.is_available()):            p = model(input_ids=batch['input_ids'].to(args.device),                      p_ids=batch['p_ids'].to(args.device),                      attn_mask=batch['attn_mask'].to(args.device))        probs.extend(p[:, 1].float().cpu().numpy())        labels.extend(batch['label'].numpy())    probs = np.asarray(probs, dtype=np.float64)    labels = np.asarray(labels, dtype=np.int64)    # ">" not ">=", matching the text-arm seed notebooks exactly    acc = float(((probs > 0.5).astype(int) == labels).mean())    return acc, probs, labelsconfig = RobertaConfig.from_pretrained(args.model_name_or_path)config.num_labels = 2encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)model = Model(encoder, config).to(args.device)print(f"Cold start from {args.model_name_or_path} -- no fine-tuned weights loaded.")train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds),                          batch_size=args.train_batch_size,                          num_workers=args.num_workers, drop_last=True)no_decay = ["bias", "LayerNorm.weight"]optimizer = AdamW([    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],     "weight_decay": 0.01},    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},], lr=args.learning_rate, eps=1e-8)updates_per_epoch = len(train_loader) // args.gradient_accumulation_stepsif len(train_loader) % args.gradient_accumulation_steps != 0:    updates_per_epoch += 1total_steps = updates_per_epoch * args.num_train_epochs# 10% linear warmup, as in the DFG training notebook. An earlier version of the# TEXT seed notebooks passed 0 here and cost three runs; do not change it.scheduler = get_linear_schedule_with_warmup(    optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)scaler = GradScaler("cuda", enabled=torch.cuda.is_available())best_val, best_state, patience_counter, history = -1.0, None, 0, []epoch_h = []   # per-epoch durations, for the predictive budget check belowt0 = time.time()stop_reason = "completed all epochs"for epoch in range(args.num_train_epochs):    model.train()    running = 0.0    optimizer.zero_grad(set_to_none=True)    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch+1}/{args.num_train_epochs}")):        with autocast(device_type="cuda", enabled=torch.cuda.is_available()):            loss, _ = model(input_ids=batch['input_ids'].to(args.device),                            p_ids=batch['p_ids'].to(args.device),                            attn_mask=batch['attn_mask'].to(args.device),                            labels=batch['label'].to(args.device))            loss = loss / args.gradient_accumulation_steps        scaler.scale(loss).backward()        if (step + 1) % args.gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):            scaler.unscale_(optimizer)            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)            scaler.step(optimizer); scaler.update(); scheduler.step()            optimizer.zero_grad(set_to_none=True)        running += loss.item() * args.gradient_accumulation_steps    val_acc, _, _ = evaluate(model, val_ds, "val")    elapsed = (time.time() - t0) / 3600.0    history.append({'epoch': epoch + 1, 'train_loss': running / max(1, len(train_loader)),                    'val_acc': val_acc, 'elapsed_h': elapsed})    print(f"  epoch {epoch+1}: train_loss={running/max(1,len(train_loader)):.4f} "          f"val_acc={val_acc*100:.4f}%  ({elapsed:.2f}h)")    if val_acc > best_val:        best_val, patience_counter = val_acc, 0        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}        print("    new best")    else:        patience_counter += 1        print(f"    no improvement ({patience_counter}/{args.patience})")        if patience_counter >= args.patience:            stop_reason = f"early stopping at epoch {epoch+1}"            print(f"  -> {stop_reason}"); break    # Predictive, not retrospective. Kaggle kills the session at 12h and writes    # nothing, so the question is not "have we passed the budget" but "would one    # more epoch pass it". Estimated with the slowest epoch seen so far, which is    # the conservative choice. A backward-looking check can pass at 10.2h and then    # lose the whole run to an epoch that ends at 11.9h.    epoch_h.append(elapsed - sum(epoch_h))    projected = elapsed + max(epoch_h)    if projected > args.time_budget_hours:        stop_reason = (f"time budget at epoch {epoch+1}: elapsed {elapsed:.2f}h, "                       f"another epoch would reach {projected:.2f}h")        print(f"  -> {stop_reason}"); breakif best_state is not None:    model.load_state_dict(best_state)print(f"\nDone: {stop_reason}. Best val accuracy {best_val*100:.4f}%")

In [ ]:
test_acc, test_probs, test_labels = evaluate(model, test_ds, "final test")def auc(y, p):    order = np.argsort(p)    ranks = np.empty(len(p), dtype=np.float64)    ranks[order] = np.arange(1, len(p) + 1)    pos, neg = y.sum(), (1 - y).sum()    return float((ranks[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))tp = int(((test_probs > 0.5) & (test_labels == 1)).sum())fp = int(((test_probs > 0.5) & (test_labels == 0)).sum())fn = int(((test_probs <= 0.5) & (test_labels == 1)).sum())prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)f1 = 2 * prec * rec / max(1e-12, prec + rec)roc = auc(test_labels, test_probs)print(f"\nSEED {args.seed}  (DFG arm)")print(f"  Accuracy : {test_acc*100:.4f}%")print(f"  ROC-AUC  : {roc:.4f}")print(f"  F1       : {f1:.4f}")print(f"  FN / FP  : {fn:,} / {fp:,}")print(f"  Test set : {len(test_labels):,}")print("\n  Context only, NOT a pass/fail check:")print(f"    Table 2 GCB+DFG : 87.8593%  FN 1,054  FP 1,197")print(f"    this run        : {test_acc*100:.4f}%  FN {fn:,}  FP {fp:,}")print(f"    delta           : {test_acc*100-87.8593:+.4f}pp")print(f"    The text arm's five runs spanned 0.4045pp. A gap of that order here")print(f"    is expected and is the quantity this experiment exists to measure.")

In [ ]:
out = {    'seed': args.seed,    'arm': 'dfg',    'accuracy': test_acc, 'roc_auc': roc, 'f1': f1,    'false_negatives': fn, 'false_positives': fp,    'test_set_size': int(len(test_labels)),    'duplicate_filtered': True,    'best_val_accuracy': best_val,    'stop_reason': stop_reason,    'epochs_run': len(history),    'config': {        'start_from': args.model_name_or_path,        'cold_start': True,        'code_length': args.code_length,        'data_flow_length': args.data_flow_length,        'train_batch_size': args.train_batch_size,        'gradient_accumulation_steps': args.gradient_accumulation_steps,        'effective_batch_size': args.train_batch_size * args.gradient_accumulation_steps,        'num_train_epochs': args.num_train_epochs,        'patience': args.patience,        'learning_rate': args.learning_rate,        'warmup_ratio': 0.1,        'split_seed': args.split_seed,        'cudnn_deterministic': False,    },    'history': history,}with open(f'/kaggle/working/test3b_dfgseed{args.seed}_results.json', 'w') as f:    json.dump(out, f, indent=2)with open(f'/kaggle/working/test3b_dfgseed{args.seed}_results.txt', 'w') as f:    f.write(f"Test 3b: Multi-Seed Robustness, DFG arm -- seed {args.seed}\n")    f.write("=" * 62 + "\n")    f.write("Model      : GraphCodeBERT + DFG-aware attention\n")    f.write(f"Start from : {args.model_name_or_path}  (cold start, public weights)\n")    f.write(f"Budget     : {args.code_length} code + {args.data_flow_length} DFG nodes\n")    f.write(f"Batch      : {args.train_batch_size} x {args.gradient_accumulation_steps}"            f" = {args.train_batch_size*args.gradient_accumulation_steps} effective\n")    f.write(f"Training   : {args.num_train_epochs} epochs / patience {args.patience},"            f" lr {args.learning_rate}, 10% warmup\n")    f.write(f"Test set   : {len(test_labels):,} samples, duplicate-filtered\n")    f.write(f"Stopped    : {stop_reason} (after {len(history)} epochs)\n\n")    f.write(f"Accuracy   : {test_acc*100:.4f}%\n")    f.write(f"ROC-AUC    : {roc:.4f}\n")    f.write(f"F1         : {f1:.4f}\n")    f.write(f"FN / FP    : {fn:,} / {fp:,}\n\n")    f.write("Per-epoch validation accuracy:\n")    for h in history:        f.write(f"  epoch {h['epoch']:>2}: val_acc={h['val_acc']*100:.4f}%  "                f"train_loss={h['train_loss']:.4f}  ({h['elapsed_h']:.2f}h)\n")    f.write("\nProvenance: test_scripts/test_3b_dfg_multiseed/\n")np.save(f'/kaggle/working/test3b_dfgseed{args.seed}_probs.npy', test_probs)print("Saved:")for s in ('results.json', 'results.txt', 'probs.npy'):    print(f"  /kaggle/working/test3b_dfgseed{args.seed}_{s}")print("\nRun seed 2025 next.")